In [1]:
import dagshub
import mlflow

dagshub.init(repo_owner="shreychauhan02", repo_name="my-mlflow-project", mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow")
mlflow.set_experiment('Ml Algos with HP Tuning')

Accessing as shreychauhan02

Initialized MLflow to track repo "shreychauhan02/my-mlflow-project"

Repository shreychauhan02/my-mlflow-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/a95761f57a3c450bb153e5ef364e8c8a', creation_time=1790045681840, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1790045681840, lifecycle_stage='active', name='Ml Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [2]:

# ============================================================
# 3. Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from imblearn.over_sampling import ADASYN
import optuna
import mlflow.sklearn


In [3]:

df = pd.read_csv("../dataset/pre.csv").dropna(subset=['clean_comment'])
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Map category if needed (same as your XGBoost notebook)
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['category'])

X = df['clean_comment']
y = df['category']

# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp
)

Shape: (36662, 2)
Columns: ['clean_comment', 'category']


In [4]:
def vectorize_data(X_train, X_val, X_test, max_features=1000, ngram_range=(1, 2)):
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec   = vectorizer.transform(X_val)
    X_test_vec  = vectorizer.transform(X_test)
    
    return X_train_vec, X_val_vec, X_test_vec, vectorizer

X_train_vec, X_val_vec, X_test_vec, vectorizer = vectorize_data(
    X_train, X_val, X_test, max_features=1000, ngram_range=(1, 2)
)

In [5]:
# ============================================================
# 6. Resampling (ADASYN)
# ============================================================
adasyn = ADASYN(random_state=42)
X_resampled, y_resampled = adasyn.fit_resample(X_train_vec, y_train)

print("After ADASYN:", X_resampled.shape)

After ADASYN: (33627, 1000)


In [6]:

# ============================================================
# 7. Shared Logging Function (same style as your XGBoost notebook)
# ============================================================
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("resampling_technique", "ADASYN")
        mlflow.set_tag("vectorizer_type", "TF-IDF")
        
        mlflow.log_param("algo_name", model_name)
        
        # Train
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)
        
        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix - {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()
        
        # Log model
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        
        print(f" {model_name} logged | Accuracy: {accuracy:.4f}")



In [7]:
# ============================================================
# 8. Random Forest
# ============================================================
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=25),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }
    model = RandomForestClassifier(**params)
    model.fit(X_resampled, y_resampled)
    return accuracy_score(y_val, model.predict(X_val_vec))

def run_rf():
    study = optuna.create_study(direction="maximize", study_name="RandomForest")
    study.optimize(objective_rf, n_trials=30)
    
    print("Best RF params:", study.best_params)
    
    best_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)
    log_mlflow("RandomForest", best_model, X_resampled, X_test_vec, y_resampled, y_test)

In [8]:
# ============================================================
# 9. Logistic Regression (Updated for sklearn 1.8+)
# ============================================================
def objective_lr(trial):
    params = {
        "C": trial.suggest_float("C", 1e-3, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),  # 0 = pure L2, 1 = pure L1
        "solver": "saga",
        "random_state": 42,
        "max_iter": 2000
    }
    model = LogisticRegression(**params)
    model.fit(X_resampled, y_resampled)
    return accuracy_score(y_val, model.predict(X_val_vec))

def run_lr():
    study = optuna.create_study(direction="maximize", study_name="LogisticRegression")
    study.optimize(objective_lr, n_trials=30)
    
    print("Best LR params:", study.best_params)
    
    best_model = LogisticRegression(
        **study.best_params,
        solver="saga",
        random_state=42,
        max_iter=2000
    )
    log_mlflow("LogisticRegression", best_model, X_resampled, X_test_vec, y_resampled, y_test)

In [9]:
# ============================================================
# 10. Multinomial Naive Bayes
# ============================================================
def objective_nb(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-3, 2.0, log=True),
        "fit_prior": trial.suggest_categorical("fit_prior", [True, False])
    }
    model = MultinomialNB(**params)
    model.fit(X_resampled, y_resampled)
    return accuracy_score(y_val, model.predict(X_val_vec))

def run_nb():
    study = optuna.create_study(direction="maximize", study_name="MultinomialNB")
    study.optimize(objective_nb, n_trials=30)
    
    print("Best NB params:", study.best_params)
    
    best_model = MultinomialNB(**study.best_params)
    log_mlflow("MultinomialNB", best_model, X_resampled, X_test_vec, y_resampled, y_test)

In [ ]:
# ============================================================
# 11. SVM (Improved)
# ============================================================
def objective_svm(trial):
    params = {
        "C": trial.suggest_float("C", 1e-2, 30, log=True),
        "kernel": trial.suggest_categorical("kernel", ["linear", "rbf"]),
        "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),
        "random_state": 42
    }
    model = SVC(**params)
    model.fit(X_resampled, y_resampled)
    return accuracy_score(y_val, model.predict(X_val_vec))

def run_svm():
    study = optuna.create_study(direction="maximize", study_name="SVM")
    study.optimize(objective_svm, n_trials=15)   # 15 trials enough, warna bahut time lega
    
    print("Best SVM params:", study.best_params)
    
    best_model = SVC(**study.best_params, random_state=42)
    log_mlflow("SVM", best_model, X_resampled, X_test_vec, y_resampled, y_test)

In [11]:

# run_rf()

In [12]:
run_lr()


[I 2026-09-24 08:47:54,903] A new study created in memory with name: LogisticRegression
[I 2026-09-24 08:47:58,273] Trial 0 finished with value: 0.7775959265320967 and parameters: {'C': 0.2363759959539951, 'l1_ratio': 0.6430343139131072}. Best is trial 0 with value: 0.7775959265320967.
[I 2026-09-24 08:48:07,809] Trial 1 finished with value: 0.7657755955628296 and parameters: {'C': 0.4745249120757361, 'l1_ratio': 0.07608739258117969}. Best is trial 0 with value: 0.7775959265320967.
[I 2026-09-24 08:48:45,766] Trial 2 finished with value: 0.773413347881433 and parameters: {'C': 9.882622892614014, 'l1_ratio': 0.6865243950353073}. Best is trial 0 with value: 0.7775959265320967.
[I 2026-09-24 08:49:47,612] Trial 3 finished with value: 0.7726859428987088 and parameters: {'C': 28.632476198339514, 'l1_ratio': 0.6881025183858814}. Best is trial 0 with value: 0.7775959265320967.
[I 2026-09-24 08:49:56,423] Trial 4 finished with value: 0.7655937443171486 and parameters: {'C': 0.41842839643091717

Best LR params: {'C': 0.8915845654693937, 'l1_ratio': 0.892660559433331}


2026/09/24 08:57:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


 LogisticRegression logged | Accuracy: 0.7846
🏃 View run LogisticRegression_ADASYN_TFIDF at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/4/runs/87e55d020b754740b05272d9eac8fa38
🧪 View experiment at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/4


In [13]:
run_nb()


[I 2026-09-24 08:58:12,531] A new study created in memory with name: MultinomialNB
[I 2026-09-24 08:58:12,586] Trial 0 finished with value: 0.7135842880523732 and parameters: {'alpha': 0.009412085135080128, 'fit_prior': False}. Best is trial 0 with value: 0.7135842880523732.
[I 2026-09-24 08:58:12,596] Trial 1 finished with value: 0.7135842880523732 and parameters: {'alpha': 0.009819323497065227, 'fit_prior': False}. Best is trial 0 with value: 0.7135842880523732.
[I 2026-09-24 08:58:12,605] Trial 2 finished with value: 0.7135842880523732 and parameters: {'alpha': 0.009194104488268759, 'fit_prior': False}. Best is trial 0 with value: 0.7135842880523732.
[I 2026-09-24 08:58:12,615] Trial 3 finished with value: 0.7123113293326059 and parameters: {'alpha': 0.06648827992113472, 'fit_prior': False}. Best is trial 0 with value: 0.7135842880523732.
[I 2026-09-24 08:58:12,636] Trial 4 finished with value: 0.7123113293326059 and parameters: {'alpha': 0.04764479284205455, 'fit_prior': False}. Be

Best NB params: {'alpha': 0.009412085135080128, 'fit_prior': False}


2026/09/24 08:59:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


 MultinomialNB logged | Accuracy: 0.7079
🏃 View run MultinomialNB_ADASYN_TFIDF at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/4/runs/a257b7ff3dd34c5fb12fdfb8ed06ff6b
🧪 View experiment at: https://dagshub.com/shreychauhan02/my-mlflow-project.mlflow/#/experiments/4


In [ ]:
run_svm()

[I 2026-09-24 08:59:34,976] A new study created in memory with name: SVM
[I 2026-09-24 09:01:48,303] Trial 0 finished with value: 0.7501363884342608 and parameters: {'C': 0.09230174332787781, 'kernel': 'linear'}. Best is trial 0 with value: 0.7501363884342608.
[I 2026-09-24 09:03:57,072] Trial 1 finished with value: 0.7579559919985452 and parameters: {'C': 0.1294485872754924, 'kernel': 'linear'}. Best is trial 1 with value: 0.7579559919985452.
[I 2026-09-24 09:06:33,477] Trial 2 finished with value: 0.6652118567012184 and parameters: {'C': 0.12984134553106658, 'kernel': 'rbf'}. Best is trial 1 with value: 0.7579559919985452.
[I 2026-09-24 09:13:05,063] Trial 3 finished with value: 0.7583196944899072 and parameters: {'C': 13.706859525850126, 'kernel': 'rbf'}. Best is trial 3 with value: 0.7583196944899072.
[I 2026-09-24 09:16:36,503] Trial 4 finished with value: 0.7663211492998727 and parameters: {'C': 0.3030080389731281, 'kernel': 'rbf'}. Best is trial 4 with value: 0.7663211492998727.